# Descarga de metadatos de Last.fm
En esta notebook consultamos la API de Last.fm y guardamos los resultados en archivos Parquet para usarlos después en el recomendador.

In [2]:
import os
import time
from getpass import getpass

import numpy as np
import pandas as pd
import requests

from config import CLEAN_DIR, conectar, ruta_sql


# Parámetros de la descarga
N_ARTISTAS = 1000
FECHA_VAL = '2013-10-01'
FECHA_TEST = '2014-01-01'
CONSULTAR_API = True
MAX_CANCIONES_ARTISTA = 2
ESPERA_API = 1.1  # segundos de espera entre consultas


# CLEAN_DIR apunta a data/clean. Su carpeta superior es data.
ruta_parquet = CLEAN_DIR.parent / 'parquet'
archivo_artistas = ruta_parquet / 'artists.parquet'
archivo_tracks = ruta_parquet / 'tracks.parquet'


# conectar() ya configura la memoria, la carpeta temporal y la zona horaria.
con = conectar()
con.execute("SET threads = 4")

# Para esta descarga usamos las escuchas registradas desde 2010.
con.execute(f"""
    CREATE OR REPLACE VIEW events_clean AS
    SELECT *
    FROM {ruta_sql('events_clean')}
    WHERE "timestamp" >= epoch(TIMESTAMP '2010-01-01')
""")

def q(consulta):
    """Ejecuta la consulta SQL recibida y devuelve las filas en un DataFrame."""
    return con.sql(consulta).df()


# Las fechas de corte se comparan con las escuchas, guardadas en segundos Unix.
def unix(fecha):
    """Convierte una fecha en segundos desde el 1 de enero de 1970 en UTC, la unidad de las escuchas."""
    return int(pd.Timestamp(fecha, tz='UTC').timestamp())


# Para consultar Last.fm necesitamos el nombre correspondiente a cada artist_id.
artistas = pd.read_parquet(archivo_artistas, columns=['artist_id', 'artist_name'])
artistas = artistas.dropna(subset=['artist_id']).drop_duplicates('artist_id')
nombres_artistas = artistas.set_index('artist_id')['artist_name']

print('Artistas con nombre:', nombres_artistas.notna().sum())

Artistas con nombre: 3190371


## Elegir los artistas de cada período

In [ ]:
def catalogo(corte):
    """Recibe una fecha de corte y devuelve los identificadores de los artistas con más oyentes distintos antes de esa fecha."""
    # Cada usuario cuenta una vez por artista. En caso de empate ordenamos por ID.
    return q(f"""
        SELECT artist_id FROM events_clean
        WHERE timestamp < {unix(corte)} AND artist_id IS NOT NULL AND user_id IS NOT NULL
        GROUP BY artist_id ORDER BY COUNT(DISTINCT user_id) DESC, artist_id
        LIMIT {N_ARTISTAS}
    """).artist_id

## Elegir hasta dos canciones por artista

In [5]:
def canciones_historicas(artistas, corte):
    """Recibe los artistas y una fecha de corte. Devuelve una tabla con sus canciones más escuchadas hasta antes del corte. Si faltan los nombres de las canciones, devuelve una tabla vacía."""
    if not archivo_tracks.exists():
        print('No encontré tracks.parquet')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    # Revisamos los nombres de las columnas antes de intentar unir las tablas.
    columnas = set(q(f"DESCRIBE SELECT * FROM '{archivo_tracks.as_posix()}'").column_name)
    if not {'track_id','track_name'}.issubset(columnas):
        print('tracks.parquet no tiene track_id y track_name')
        return pd.DataFrame(columns=['artist_id', 'track_id', 'track_name', 'artist_name', 'escuchas'])
    # DuckDB podrá usar esta lista de artistas como tabla dentro de la consulta.
    con.register('catalogo_canciones', pd.DataFrame({'artist_id': np.asarray(artistas, dtype='int64')}))
    # Primero contamos escuchas por canción y después ordenamos dentro de cada artista.
    canciones = q(f"""
        WITH conteo AS (
            SELECT e.artist_id, e.track_id, COUNT(*) AS escuchas
            FROM events_clean e JOIN catalogo_canciones c USING(artist_id)
            WHERE e.timestamp < {unix(corte)} AND e.track_id IS NOT NULL
            GROUP BY e.artist_id, e.track_id
        ), orden AS (
            SELECT *, ROW_NUMBER() OVER(
                PARTITION BY artist_id ORDER BY escuchas DESC, track_id
            ) AS puesto
            FROM conteo
        )
        SELECT o.artist_id, o.track_id, t.track_name, o.escuchas
        FROM orden o JOIN '{archivo_tracks.as_posix()}' t USING(track_id)
        WHERE o.puesto <= {MAX_CANCIONES_ARTISTA}
    """)
    # Ya tenemos el resultado en pandas y podemos quitar la tabla temporal registrada.
    con.unregister('catalogo_canciones')
    # Buscamos el nombre de cada artista por su identificador para consultar Last.fm.
    canciones['artist_name'] = canciones.artist_id.map(nombres_artistas)
    return canciones.dropna(subset=['artist_name', 'track_name'])

## Archivos donde guardamos las descargas

In [6]:
# Reutilizamos las etiquetas guardadas en descargas anteriores.
ARCHIVO_TAGS = ruta_parquet / 'artist_tags_lastfm_enriquecido.parquet'
ARCHIVO_TAGS_ESTADOS = ruta_parquet / 'artist_tags_lastfm_estados.parquet'
ARCHIVO_TAGS_ESTADOS = ruta_parquet / 'artist_tags_lastfm_estados.parquet'
CACHE_INFO = ruta_parquet / 'artist_info_actual_lastfm.parquet'
CACHE_SIM = ruta_parquet / 'artist_similares_actual_lastfm.parquet'
CACHE_TOP = ruta_parquet / 'artist_toptracks_actual_lastfm.parquet'
CACHE_TRACKS = ruta_parquet / 'track_info_y_tags_actual_lastfm.parquet'
URL_LASTFM = 'https://ws.audioscrobbler.com/2.0/'
API_KEY = None

## Hacer una consulta a Last.fm

In [7]:
def api(metodo, **kwargs):
    """Recibe el método de Last.fm y sus parámetros, por ejemplo el nombre del artista. Devuelve la respuesta como diccionario. Para el código 6 devuelve un diccionario vacío. Si los intentos se agotan, comunica el error."""
    # Guardamos la clave en memoria para reutilizarla en las siguientes consultas.
    global API_KEY
    if API_KEY is None:
        API_KEY = os.getenv('LASTFM_API_KEY', '').strip() or getpass('Clave API Last.fm: ').strip()
    if not API_KEY:
        raise RuntimeError('No se indicó la clave API')
    # kwargs contiene los parámetros propios de cada método, como artist o track.
    params = dict(method=metodo, api_key=API_KEY, format='json', **kwargs)
    # Hacemos hasta cuatro intentos. Entre errores esperamos 3, 6 y 12 segundos.
    for intento in range(4):
        try:
            r = requests.get(URL_LASTFM, params=params, timeout=25)
            # El servidor puede pedir que bajemos el ritmo o estar temporalmente ocupado.
            if r.status_code in [429, 500, 502, 503, 504]:
                raise requests.RequestException(f'HTTP {r.status_code}')
            r.raise_for_status()
            datos = r.json()
            if 'error' in datos:
                if int(datos['error']) == 6:
                    return {}  # Artista/canción no encontrada.
                if int(datos['error']) in [8, 11, 16, 29]:
                    raise requests.RequestException(f"API {datos['error']}: {datos.get('message')}")
                raise ValueError(f"Error Last.fm {datos['error']}: {datos.get('message')}")
            time.sleep(ESPERA_API)
            return datos
        except (requests.RequestException, ValueError):
            if intento == 3:
                raise
            time.sleep(min(30, 3 * (2 ** intento)))

## Preparar las respuestas y leer los archivos guardados

In [8]:
def lista(x):
    """Convierte la respuesta de Last.fm en una lista para poder recorrerla. Un diccionario representa un solo resultado. Si falta el dato, devuelve una lista vacía."""
    if isinstance(x, list):
        return x
    return [x] if isinstance(x, dict) else []

In [9]:
def numero(x):
    """Convierte un valor recibido como texto en un número decimal. Si falta o no se puede convertir, usa 0.0 como valor de reemplazo."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0

In [10]:
def leer_cache(archivo, columnas):
    """Lee un Parquet guardado y comprueba que tenga las columnas pedidas. Si el archivo o alguna columna falta, devuelve una tabla vacía con esos nombres."""
    if archivo.exists():
        df = pd.read_parquet(archivo)
        if set(columnas).issubset(df):
            return df
    return pd.DataFrame(columns=columnas)

## Descargar las etiquetas de los artistas

In [ ]:
def descargar_tags(ids):
    """Recibe identificadores de artistas. Recupera las etiquetas guardadas, consulta las pendientes cuando la descarga está habilitada y devuelve la tabla de etiquetas y pesos."""
    columnas = ['artist_id', 'artist_name', 'tag', 'peso']
    tags = leer_cache(ARCHIVO_TAGS, columnas)
    estados = leer_cache(ARCHIVO_TAGS_ESTADOS, ['artist_id', 'estado'])
    # Tanto una respuesta con etiquetas como una respuesta sin etiquetas cuentan como procesadas.
    conocidos = set(estados.loc[estados.estado.isin(['ok', 'sin_tags']), 'artist_id'].astype(int))
    # Si existe una tabla de etiquetas previa pero no sus estados, no repetir lo ya disponible.
    conocidos |= set(tags.artist_id.dropna().astype(int))
    pendientes = [int(i) for i in ids if int(i) not in conocidos]
    print('Etiquetas pendientes:', len(pendientes))
    if not CONSULTAR_API:
        return tags
    for i, aid in enumerate(pendientes, 1):
        nombre = nombres_artistas.get(aid)
        if not isinstance(nombre, str):
            continue
        try:
            datos = api('artist.getTopTags', artist=nombre)
            # Cada etiqueta se guarda en una fila junto con su artista y el peso informado por Last.fm.
            nuevas = [dict(artist_id=aid, artist_name=nombre,
                           tag=str(x.get('name','')).strip().casefold(), peso=numero(x.get('count')))
                      for x in lista(datos.get('toptags', {}).get('tag'))]
            nuevas = [x for x in nuevas if x['tag'] and x['peso'] > 0]
            if nuevas:
                tags = pd.concat([tags, pd.DataFrame(nuevas)], ignore_index=True)
            estados = pd.concat([estados, pd.DataFrame([dict(artist_id=aid,
                estado='ok' if nuevas else 'sin_tags')])], ignore_index=True)
        except Exception as e:
            print('Falló la consulta de etiquetas:', nombre, str(e)[:100])
        # Guardamos el avance cada diez elementos y al llegar al último.
        if i % 10 == 0 or i == len(pendientes):
            tags = tags.drop_duplicates(['artist_id', 'tag'], keep='last')
            estados = estados.drop_duplicates('artist_id', keep='last')
            tags.to_parquet(ARCHIVO_TAGS, index=False)
            estados.to_parquet(ARCHIVO_TAGS_ESTADOS, index=False)
            print(f'Etiquetas: {i}/{len(pendientes)}')
    return tags

## Descargar estadísticas, artistas similares y canciones destacadas

In [ ]:
def descargar_artistas(ids):
    """Recibe identificadores de artistas y devuelve tres tablas con estadísticas, artistas similares y canciones destacadas. Los datos de Last.fm corresponden al momento de la consulta."""
    info = leer_cache(CACHE_INFO, ['artist_id', 'listeners_actual', 'playcount_actual'])
    similares = leer_cache(CACHE_SIM, ['artist_id', 'similar_name', 'similitud'])
    top = leer_cache(CACHE_TOP, ['artist_id', 'track_name', 'puesto_actual'])
    # Usamos los artistas presentes en la tabla de estadísticas para saber cuáles ya se consultaron.
    hechos = set(info.artist_id.dropna().astype(int))
    pendientes = [int(i) for i in ids if int(i) not in hechos]
    print('Metadatos de artistas pendientes:', len(pendientes))
    if not CONSULTAR_API:
        return info, similares, top
    for i, aid in enumerate(pendientes, 1):
        nombre = nombres_artistas.get(aid)
        if not isinstance(nombre, str):
            continue
        # Solo marcamos como completo un artista si se procesaron las tres consultas.
        try:
            # Para cada artista pedimos sus estadísticas, hasta 20 similares y hasta 10 canciones.
            d_info = api('artist.getInfo', artist=nombre, autocorrect=1).get('artist', {})
            d_sim = api('artist.getSimilar', artist=nombre, limit=20, autocorrect=1)
            d_top = api('artist.getTopTracks', artist=nombre, limit=10, page=1, autocorrect=1)
            stats = d_info.get('stats', {})
            info = pd.concat([info, pd.DataFrame([{
                'artist_id': aid, 'artist_name': nombre,
                'listeners_actual': numero(stats.get('listeners')),
                'playcount_actual': numero(stats.get('playcount')),
            }])], ignore_index=True)
            similares_nuevos = [{
                'artist_id': aid, 'similar_name': s.get('name', ''),
                'similitud': numero(s.get('match'))
            } for s in lista(d_sim.get('similarartists', {}).get('artist'))]
            if similares_nuevos:
                similares = pd.concat([similares, pd.DataFrame(similares_nuevos)], ignore_index=True)
            tops = [{
                'artist_id': aid, 'track_name': t.get('name', ''),
                'puesto_actual': j, 'playcount_actual': numero(t.get('playcount')),
            } for j,t in enumerate(lista(d_top.get('toptracks', {}).get('track')),1)]
            if tops:
                top = pd.concat([top, pd.DataFrame(tops)], ignore_index=True)
        except Exception as e:
            print('No se pudo completar:', nombre, str(e)[:120])
        # Guardamos el avance cada diez elementos y al llegar al último.
        if i % 10 == 0 or i == len(pendientes):
            info = info.drop_duplicates('artist_id', keep='last')
            similares = similares.drop_duplicates(['artist_id','similar_name'], keep='last')
            top = top.drop_duplicates(['artist_id','track_name'], keep='last')
            info.to_parquet(CACHE_INFO, index=False)
            similares.to_parquet(CACHE_SIM, index=False)
            top.to_parquet(CACHE_TOP, index=False)
            print(f'Metadatos de artistas: {i}/{len(pendientes)}')
    return info, similares, top

## Descargar los datos de las canciones elegidas

In [ ]:
def descargar_canciones(candidatas):
    """Recibe una tabla de canciones con sus nombres e identificadores. Devuelve los datos guardados y, si la descarga está habilitada, incorpora duración, oyentes y etiquetas de las canciones pendientes."""
    info = leer_cache(CACHE_TRACKS, ['artist_id', 'track_id', 'track_name', 'duracion_ms',
                                    'listeners_actual', 'tags_json'])
    # Tomamos los identificadores de canciones que ya figuran en el Parquet.
    hechos = set(pd.to_numeric(info.track_id, errors='coerce').dropna().astype(int))
    pendientes = candidatas[~candidatas.track_id.isin(hechos)]
    print('Canciones pendientes:', len(pendientes))
    if not CONSULTAR_API:
        return info
    filas = []
    import json
    for i, t in enumerate(pendientes.itertuples(index=False), 1):
        try:
            # Consultamos por nombre de artista y canción. Conservamos los identificadores locales para el cruce posterior.
            d = api('track.getInfo', artist=t.artist_name, track=t.track_name, autocorrect=1).get('track', {})
            e = api('track.getTopTags', artist=t.artist_name, track=t.track_name, autocorrect=1)
            tags = [{'tag': x.get('name', '').strip().casefold(),
                     'peso': numero(x.get('count'))}
                    for x in lista(e.get('toptags', {}).get('tag'))]
            # Guardamos las etiquetas de una canción como texto JSON dentro de una sola celda.
            filas.append({'artist_id': int(t.artist_id), 'track_id': int(t.track_id),
                          'track_name': t.track_name,
                          'duracion_ms': numero(d.get('duration')),
                          'listeners_actual': numero(d.get('listeners')),
                          'tags_json': json.dumps(tags, ensure_ascii=False)})
        except Exception as error:
            print('Canción sin respuesta:', t.track_name, str(error)[:100])
        # Guardamos el avance cada diez elementos y al llegar al último.
        if i % 10 == 0 or i == len(pendientes):
            if filas:
                info = pd.concat([info, pd.DataFrame(filas)], ignore_index=True)
            info = info.drop_duplicates('track_id', keep='last')
            info.to_parquet(CACHE_TRACKS, index=False)
            filas = []
            print(f'Canciones: {i}/{len(pendientes)}')
    return info

## Ejecutar la descarga

In [ ]:
catalogos = {fecha: catalogo(fecha) for fecha in (FECHA_VAL, FECHA_TEST)}
# Un artista puede aparecer en los dos cortes. Lo consultamos una sola vez.
ids = sorted(set().union(*catalogos.values()))
# Elegimos las canciones por separado para cada corte y luego quitamos las repetidas.
canciones = pd.concat([canciones_historicas(ids_corte, fecha)
                       for fecha, ids_corte in catalogos.items()]).drop_duplicates('track_id')
# Acá se llaman las funciones definidas arriba y se inicia la descarga si hay pendientes.
tags = descargar_tags(ids)
info, similares, top = descargar_artistas(ids)
tracks = descargar_canciones(canciones)
print('Catálogo combinado:', len(ids), '| Con etiquetas:', tags.artist_id.isin(ids).groupby(tags.artist_id).any().sum())